In [1]:
import os
import pandas as pd
import numpy as np
os.makedirs("data", exist_ok=True)
os.makedirs("src", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Folders created.")

Folders created.


In [2]:
np.random.seed(42)

n = 200

data = {
    "order_id": list(range(1, n+1)),
    "price": np.random.randint(10, 500, n),
    "quantity": np.random.randint(1, 10, n),
    "customer_rating": np.random.uniform(1, 5, n),
    "delivery_days": np.random.randint(1, 15, n),
    "category": np.random.choice(["electronics", "clothing", "home"], n)
}

df = pd.DataFrame(data)

# Inject issues
df.loc[5, "price"] = -50
df.loc[10, "quantity"] = np.nan
df.loc[20, "customer_rating"] = 7
df.loc[30, "category"] = "unknown"
df.loc[40, "delivery_days"] = -3

# Duplicate row
df = pd.concat([df, df.iloc[[1]]], ignore_index=True)

df.to_csv("data/audit_dataset.csv", index=False)

print("Dataset created successfully.")
df.head()

Dataset created successfully.


,order_id,price,quantity,customer_rating,delivery_days,category
0,1,112,4.0,1.283764,3,electronics
1,2,445,7.0,2.587135,4,home
2,3,358,3.0,1.203074,9,home
3,4,280,6.0,4.546469,2,home
4,5,116,2.0,1.110467,9,electronics


In [7]:
with open("src/__init.py", "w") as f:
  f.write("")

In [3]:
%%writefile src/audit_rules.py

def check_missing_values(df):
    return df.isnull().sum().to_dict()

def check_negative_values(df):
    issues = {}
    for col in ["price", "quantity", "delivery_days"]:
        issues[col] = int((df[col] < 0).sum())
    return issues

def check_invalid_ranges(df):
    issues = {}
    issues["customer_rating_invalid"] = int(((df["customer_rating"] < 1) | (df["customer_rating"] > 5)).sum())
    return issues

def check_invalid_categories(df):
    valid = ["electronics", "clothing", "home"]
    return int((~df["category"].isin(valid)).sum())

def check_duplicates(df):
    return int(df.duplicated().sum())

Writing src/audit_rules.py


In [4]:
import sys
sys.path.append("/content")

from src.audit_rules import *

df = pd.read_csv("data/audit_dataset.csv")

print("Missing:", check_missing_values(df))
print("Negative:", check_negative_values(df))
print("Invalid range:", check_invalid_ranges(df))
print("Invalid category:", check_invalid_categories(df))
print("Duplicates:", check_duplicates(df))

Missing: {'order_id': 0, 'price': 0, 'quantity': 1, 'customer_rating': 0, 'delivery_days': 0, 'category': 0}
Negative: {'price': 1, 'quantity': 0, 'delivery_days': 1}
Invalid range: {'customer_rating_invalid': 1}
Invalid category: 1
Duplicates: 1


In [5]:
%%writefile src/baseline_auditor.py

from src.audit_rules import *

def run_baseline_audit(df):
    report = {
        "missing_values": check_missing_values(df),
        "negative_values": check_negative_values(df),
        "invalid_ranges": check_invalid_ranges(df),
        "invalid_categories": check_invalid_categories(df),
        "duplicates": check_duplicates(df)
    }
    return report

Writing src/baseline_auditor.py


In [6]:
from src.baseline_auditor import run_baseline_audit
df = pd.read_csv("data/audit_dataset.csv")
baseline_report = run_baseline_audit(df)
baseline_report

{'missing_values': {'order_id': 0,
  'price': 0,
  'quantity': 1,
  'customer_rating': 0,
  'delivery_days': 0,
  'category': 0},
 'negative_values': {'price': 1, 'quantity': 0, 'delivery_days': 1},
 'invalid_ranges': {'customer_rating_invalid': 1},
 'invalid_categories': 1,
 'duplicates': 1}

In [7]:
%%writefile src/mcp_tools.py
import pandas as pd
from src.audit_rules import (
    check_missing_values,
    check_negative_values,
    check_invalid_ranges,
    check_invalid_categories,
    check_duplicates,
)

def read_data(file_path="data/audit_dataset.csv"):
    df = pd.read_csv(file_path)
    return {
        "status": "success",
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": df.columns.tolist()
    }

def run_missing_value_check(df):
    return {
        "tool_name": "run_missing_value_check",
        "result": check_missing_values(df)
    }

def run_negative_value_check(df):
    return {
        "tool_name": "run_negative_value_check",
        "result": check_negative_values(df)
    }

def run_invalid_range_check(df):
    return {
        "tool_name": "run_invalid_range_check",
        "result": check_invalid_ranges(df)
    }

def run_invalid_category_check(df):
    return {
        "tool_name": "run_invalid_category_check",
        "result": {
            "invalid_category_count": check_invalid_categories(df)
        }
    }

def run_duplicate_check(df):
    return {
        "tool_name": "run_duplicate_check",
        "result": {
            "duplicate_count": check_duplicates(df)
        }
    }

Writing src/mcp_tools.py


In [8]:
%%writefile src/mcp_agent.py
import pandas as pd
from src.mcp_tools import (
    read_data,
    run_missing_value_check,
    run_negative_value_check,
    run_invalid_range_check,
    run_invalid_category_check,
    run_duplicate_check,
)

class MCPAutonomousDataAuditor:
    def __init__(self, file_path="data/audit_dataset.csv"):
        self.file_path = file_path
        self.tool_calls = []

    def _record_tool_call(self, tool_name):
        self.tool_calls.append(tool_name)

    def run_audit(self):
        df = pd.read_csv(self.file_path)

        summary = read_data(self.file_path)
        self._record_tool_call("read_data")

        missing_report = run_missing_value_check(df)
        self._record_tool_call("run_missing_value_check")

        negative_report = run_negative_value_check(df)
        self._record_tool_call("run_negative_value_check")

        range_report = run_invalid_range_check(df)
        self._record_tool_call("run_invalid_range_check")

        category_report = run_invalid_category_check(df)
        self._record_tool_call("run_invalid_category_check")

        duplicate_report = run_duplicate_check(df)
        self._record_tool_call("run_duplicate_check")

        final_report = {
            "dataset_summary": summary,
            "audit_findings": {
                "missing_values": missing_report["result"],
                "negative_values": negative_report["result"],
                "invalid_ranges": range_report["result"],
                "invalid_categories": category_report["result"],
                "duplicates": duplicate_report["result"]
            },
            "tool_calls": self.tool_calls,
            "tool_call_count": len(self.tool_calls)
        }

        return final_report

Writing src/mcp_agent.py


In [9]:
import sys
sys.path.append("/content")
from src.mcp_tools import *
from src.mcp_agent import MCPAutonomousDataAuditor
print("MCP modules imported successfully.")

MCP modules imported successfully.


In [10]:
df = pd.read_csv("data/audit_dataset.csv")

print(read_data("data/audit_dataset.csv"))
print(run_missing_value_check(df))
print(run_negative_value_check(df))
print(run_invalid_range_check(df))
print(run_invalid_category_check(df))
print(run_duplicate_check(df))

{'status': 'success', 'rows': 201, 'columns': 6, 'column_names': ['order_id', 'price', 'quantity', 'customer_rating', 'delivery_days', 'category']}
{'tool_name': 'run_missing_value_check', 'result': {'order_id': 0, 'price': 0, 'quantity': 1, 'customer_rating': 0, 'delivery_days': 0, 'category': 0}}
{'tool_name': 'run_negative_value_check', 'result': {'price': 1, 'quantity': 0, 'delivery_days': 1}}
{'tool_name': 'run_invalid_range_check', 'result': {'customer_rating_invalid': 1}}
{'tool_name': 'run_invalid_category_check', 'result': {'invalid_category_count': 1}}
{'tool_name': 'run_duplicate_check', 'result': {'duplicate_count': 1}}


In [11]:
mcp_auditor = MCPAutonomousDataAuditor(file_path="data/audit_dataset.csv")
mcp_report = mcp_auditor.run_audit()

mcp_report

{'dataset_summary': {'status': 'success',
  'rows': 201,
  'columns': 6,
  'column_names': ['order_id',
   'price',
   'quantity',
   'customer_rating',
   'delivery_days',
   'category']},
 'audit_findings': {'missing_values': {'order_id': 0,
   'price': 0,
   'quantity': 1,
   'customer_rating': 0,
   'delivery_days': 0,
   'category': 0},
  'negative_values': {'price': 1, 'quantity': 0, 'delivery_days': 1},
  'invalid_ranges': {'customer_rating_invalid': 1},
  'invalid_categories': {'invalid_category_count': 1},
  'duplicates': {'duplicate_count': 1}},
 'tool_calls': ['read_data',
  'run_missing_value_check',
  'run_negative_value_check',
  'run_invalid_range_check',
  'run_invalid_category_check',
  'run_duplicate_check'],
 'tool_call_count': 6}

In [12]:
def show_mcp_report(report):
    print("=" * 80)
    print("DATASET SUMMARY")
    print(report["dataset_summary"])

    print("\nAUDIT FINDINGS")
    for key, value in report["audit_findings"].items():
        print(f"\n{key.upper()}:")
        print(value)

    print("\nTOOL CALLS:")
    print(report["tool_calls"])
    print("Tool Call Count:", report["tool_call_count"])

show_mcp_report(mcp_report)

DATASET SUMMARY
{'status': 'success', 'rows': 201, 'columns': 6, 'column_names': ['order_id', 'price', 'quantity', 'customer_rating', 'delivery_days', 'category']}

AUDIT FINDINGS

MISSING_VALUES:
{'order_id': 0, 'price': 0, 'quantity': 1, 'customer_rating': 0, 'delivery_days': 0, 'category': 0}

NEGATIVE_VALUES:
{'price': 1, 'quantity': 0, 'delivery_days': 1}

INVALID_RANGES:
{'customer_rating_invalid': 1}

INVALID_CATEGORIES:
{'invalid_category_count': 1}

DUPLICATES:
{'duplicate_count': 1}

TOOL CALLS:
['read_data', 'run_missing_value_check', 'run_negative_value_check', 'run_invalid_range_check', 'run_invalid_category_check', 'run_duplicate_check']
Tool Call Count: 6


In [13]:
comparison = {
    "baseline_report": baseline_report,
    "mcp_report": mcp_report
}
comparison

{'baseline_report': {'missing_values': {'order_id': 0,
   'price': 0,
   'quantity': 1,
   'customer_rating': 0,
   'delivery_days': 0,
   'category': 0},
  'negative_values': {'price': 1, 'quantity': 0, 'delivery_days': 1},
  'invalid_ranges': {'customer_rating_invalid': 1},
  'invalid_categories': 1,
  'duplicates': 1},
 'mcp_report': {'dataset_summary': {'status': 'success',
   'rows': 201,
   'columns': 6,
   'column_names': ['order_id',
    'price',
    'quantity',
    'customer_rating',
    'delivery_days',
    'category']},
  'audit_findings': {'missing_values': {'order_id': 0,
    'price': 0,
    'quantity': 1,
    'customer_rating': 0,
    'delivery_days': 0,
    'category': 0},
   'negative_values': {'price': 1, 'quantity': 0, 'delivery_days': 1},
   'invalid_ranges': {'customer_rating_invalid': 1},
   'invalid_categories': {'invalid_category_count': 1},
   'duplicates': {'duplicate_count': 1}},
  'tool_calls': ['read_data',
   'run_missing_value_check',
   'run_negative_valu

In [14]:
import json
with open("results/mcp_report.json", "w") as f:
  json.dump(mcp_report, f, indent=2)
print("Saved: results/mcp_report.json")

Saved: results/mcp_report.json


In [15]:
print("Project 3 — Step 2 Completed Successfully")
print("=" * 60)
print("Built:")
print("- MCP-style tool layer")
print("- Autonomous data auditing agent")
print("- Tool-call tracking")
print("- Baseline vs MCP comparison")
print("\nSaved:")
print("- results/mcp_report.json")

Project 3 — Step 2 Completed Successfully
Built:
- MCP-style tool layer
- Autonomous data auditing agent
- Tool-call tracking
- Baseline vs MCP comparison

Saved:
- results/mcp_report.json


In [17]:
ground_truth = {
    "missing_values": {
        "quantity": 1
    },
    "negative_values": {
        "price": 1,
        "delivery_days": 1
    },
    "invalid_ranges": {
        "customer_rating_invalid": 1
    },
    "invalid_categories": {
        "invalid_category_count": 1
    },
    "duplicates": {
        "duplicate_count": 1
    }
}

ground_truth

{'missing_values': {'quantity': 1},
 'negative_values': {'price': 1, 'delivery_days': 1},
 'invalid_ranges': {'customer_rating_invalid': 1},
 'invalid_categories': {'invalid_category_count': 1},
 'duplicates': {'duplicate_count': 1}}

In [16]:
%%writefile src/evaluator.py

def compute_precision_recall(predicted, actual):
    tp = 0
    fp = 0
    fn = 0

    for key in actual:
        pred_val = predicted.get(key, 0)
        true_val = actual.get(key, 0)

        if pred_val > 0 and true_val > 0:
            tp += 1
        elif pred_val > 0 and true_val == 0:
            fp += 1
        elif pred_val == 0 and true_val > 0:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall


def evaluate_audit(mcp_report, ground_truth):
    results = {}

    for category in ground_truth:
        predicted = mcp_report["audit_findings"].get(category, {})
        actual = ground_truth[category]

        precision, recall = compute_precision_recall(predicted, actual)

        results[category] = {
            "precision": round(precision, 2),
            "recall": round(recall, 2)
        }

    return results


def evaluate_tool_calls(tool_calls):
    expected_tools = {
        "read_data",
        "run_missing_value_check",
        "run_negative_value_check",
        "run_invalid_range_check",
        "run_invalid_category_check",
        "run_duplicate_check"
    }

    used_tools = set(tool_calls)

    correct_calls = len(expected_tools.intersection(used_tools))
    false_calls = len(used_tools - expected_tools)

    accuracy = correct_calls / len(expected_tools)

    return {
        "tool_call_accuracy": round(accuracy, 2),
        "false_tool_calls": false_calls
    }

Writing src/evaluator.py


In [18]:
from src.evaluator import evaluate_audit, evaluate_tool_calls
print("Evaluator loaded.")

Evaluator loaded.


In [19]:
audit_eval = evaluate_audit(mcp_report, ground_truth)
audit_eval

{'missing_values': {'precision': 1.0, 'recall': 1.0},
 'negative_values': {'precision': 1.0, 'recall': 1.0},
 'invalid_ranges': {'precision': 1.0, 'recall': 1.0},
 'invalid_categories': {'precision': 1.0, 'recall': 1.0},
 'duplicates': {'precision': 1.0, 'recall': 1.0}}

In [20]:
tool_eval = evaluate_tool_calls(mcp_report["tool_calls"])
tool_eval

{'tool_call_accuracy': 1.0, 'false_tool_calls': 0}

In [21]:
final_eval = {
    "audit_metrics": audit_eval,
    "tool_metrics": tool_eval
}
final_eval

{'audit_metrics': {'missing_values': {'precision': 1.0, 'recall': 1.0},
  'negative_values': {'precision': 1.0, 'recall': 1.0},
  'invalid_ranges': {'precision': 1.0, 'recall': 1.0},
  'invalid_categories': {'precision': 1.0, 'recall': 1.0},
  'duplicates': {'precision': 1.0, 'recall': 1.0}},
 'tool_metrics': {'tool_call_accuracy': 1.0, 'false_tool_calls': 0}}

In [22]:
def show_evaluation(eval_result):
    print("=" * 80)
    print("AUDIT METRICS")

    for category, metrics in eval_result["audit_metrics"].items():
        print(f"\n{category.upper()}")
        print("Precision:", metrics["precision"])
        print("Recall:", metrics["recall"])

    print("\nTOOL METRICS")
    print("Tool Call Accuracy:", eval_result["tool_metrics"]["tool_call_accuracy"])
    print("False Tool Calls:", eval_result["tool_metrics"]["false_tool_calls"])


show_evaluation(final_eval)

AUDIT METRICS

MISSING_VALUES
Precision: 1.0
Recall: 1.0

NEGATIVE_VALUES
Precision: 1.0
Recall: 1.0

INVALID_RANGES
Precision: 1.0
Recall: 1.0

INVALID_CATEGORIES
Precision: 1.0
Recall: 1.0

DUPLICATES
Precision: 1.0
Recall: 1.0

TOOL METRICS
Tool Call Accuracy: 1.0
False Tool Calls: 0


In [23]:
import json

with open("results/evaluation_results.json", "w") as f:
    json.dump(final_eval, f, indent=2)

print("Saved: results/evaluation_results.json")

Saved: results/evaluation_results.json


In [24]:
print(" Project 3 — Step 3 Completed")
print("=" * 60)

print("Built:")
print("- Precision / Recall evaluation")
print("- Category-level audit evaluation")
print("- Tool call accuracy evaluation")
print("- False tool call detection")

print("\nSaved:")
print("- results/evaluation_results.json")

 Project 3 — Step 3 Completed
Built:
- Precision / Recall evaluation
- Category-level audit evaluation
- Tool call accuracy evaluation
- False tool call detection

Saved:
- results/evaluation_results.json


In [25]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import json
import sys
import os

sys.path.append("/content")

from src.baseline_auditor import run_baseline_audit
from src.mcp_agent import MCPAutonomousDataAuditor
from src.evaluator import evaluate_audit, evaluate_tool_calls


st.set_page_config(page_title="MCP-Based Autonomous Data Auditor", layout="wide")


GROUND_TRUTH = {
    "missing_values": {
        "quantity": 1
    },
    "negative_values": {
        "price": 1,
        "delivery_days": 1
    },
    "invalid_ranges": {
        "customer_rating_invalid": 1
    },
    "invalid_categories": {
        "invalid_category_count": 1
    },
    "duplicates": {
        "duplicate_count": 1
    }
}


@st.cache_data
def load_data():
    return pd.read_csv("data/audit_dataset.csv")


@st.cache_resource
def run_mcp_audit():
    auditor = MCPAutonomousDataAuditor(file_path="data/audit_dataset.csv")
    return auditor.run_audit()


df = load_data()
mcp_report = run_mcp_audit()
baseline_report = run_baseline_audit(df)
audit_eval = evaluate_audit(mcp_report, GROUND_TRUTH)
tool_eval = evaluate_tool_calls(mcp_report["tool_calls"])

st.title("MCP-Based Autonomous Data Auditor")
st.markdown(
    "Autonomous data auditing system using **MCP-style tool orchestration** to inspect datasets, detect issues, and generate structured audit reports."
)

st.subheader("Dataset Preview")
st.dataframe(df.head(10), use_container_width=True)

st.subheader("Dataset Summary")
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Rows", mcp_report["dataset_summary"]["rows"])
with col2:
    st.metric("Columns", mcp_report["dataset_summary"]["columns"])
with col3:
    st.metric("Tool Calls", mcp_report["tool_call_count"])

st.markdown("---")

st.subheader("Baseline Audit Report (Without MCP)")
st.json(baseline_report)

st.subheader("MCP Audit Findings")
st.json(mcp_report["audit_findings"])

st.subheader("Tool Calls Used")
st.write(mcp_report["tool_calls"])

st.markdown("---")

st.subheader("Evaluation Metrics")

metric_rows = []
for category, metrics in audit_eval.items():
    metric_rows.append({
        "category": category,
        "precision": metrics["precision"],
        "recall": metrics["recall"]
    })

metric_df = pd.DataFrame(metric_rows)
st.dataframe(metric_df, use_container_width=True)

col4, col5 = st.columns(2)
with col4:
    st.metric("Tool Call Accuracy", tool_eval["tool_call_accuracy"])
with col5:
    st.metric("False Tool Calls", tool_eval["false_tool_calls"])

st.markdown("---")

st.subheader("Final Audit Summary")

summary_points = [
    f"Missing value issues detected: {mcp_report['audit_findings']['missing_values']}",
    f"Negative value issues detected: {mcp_report['audit_findings']['negative_values']}",
    f"Invalid range issues detected: {mcp_report['audit_findings']['invalid_ranges']}",
    f"Invalid category issues detected: {mcp_report['audit_findings']['invalid_categories']}",
    f"Duplicate issues detected: {mcp_report['audit_findings']['duplicates']}",
]

for point in summary_points:
    st.write("-", point)

st.markdown("---")

st.subheader("Why This Project Matters")
st.write(
    """
    This project demonstrates how an autonomous agent can use tools through an MCP-style interface
    to inspect structured business data, detect quality issues, and generate explainable audit reports.
    It highlights tool usage, structured reasoning, and measurable evaluation.
    """
)

Writing streamlit_app.py


In [26]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.0 MB/s eta 0:00:00


In [36]:
!streamlit run streamlit_app.py > streamlit.log 2>&1 &

In [37]:
!npm install -g localtunnel
!lt --port 8501 > 1t.log 2>&1 &

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [38]:
!cat 1t.log

your url is: https://quiet-ears-laugh.loca.lt


In [39]:
!curl ipv4.icanhazip.com

136.109.6.147


In [27]:
!cat streamlit.log | tail -n 100

cat: streamlit.log: No such file or directory


In [41]:
print("Project  Completed Successfully")
print("=" * 60)
print("Built:")
print("- Baseline auditor")
print("- MCP-style tool layer")
print("- Autonomous data auditor")
print("- Evaluation framework")
print("- Streamlit demo")
print("\nYour 3-project portfolio is now complete.")

Project  Completed Successfully
Built:
- Baseline auditor
- MCP-style tool layer
- Autonomous data auditor
- Evaluation framework
- Streamlit demo

Your 3-project portfolio is now complete.


In [28]:
!zip -r project3.zip data src results streamlit_app.py requirements.txt Readme.md

	zip warning: name not matched: requirements.txt
	zip warning: name not matched: Readme.md
  adding: data/ (stored 0%)
  adding: data/audit_dataset.csv (deflated 58%)
  adding: src/ (stored 0%)
  adding: src/__pycache__/ (stored 0%)
  adding: src/__pycache__/baseline_auditor.cpython-312.pyc (deflated 40%)
  adding: src/__pycache__/mcp_agent.cpython-312.pyc (deflated 51%)
  adding: src/__pycache__/audit_rules.cpython-312.pyc (deflated 45%)
  adding: src/__pycache__/evaluator.cpython-312.pyc (deflated 38%)
  adding: src/__pycache__/mcp_tools.cpython-312.pyc (deflated 52%)
  adding: src/audit_rules.py (deflated 55%)
  adding: src/baseline_auditor.py (deflated 56%)
  adding: src/mcp_agent.py (deflated 73%)
  adding: src/mcp_tools.py (deflated 73%)
  adding: src/evaluator.py (deflated 67%)
  adding: results/ (stored 0%)
  adding: results/evaluation_results.json (deflated 70%)
  adding: results/mcp_report.json (deflated 67%)
  adding: streamlit_app.py (deflated 64%)
